In [1]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [2]:
from google.colab import auth
auth.authenticate_user()

In [3]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - Radiology

In [4]:
# Load the data
radiology_query = """
SELECT  rad.subject_id,
        rad.note_id,
        rad.charttime,
        rad.storetime,
        det.field_name,
        det.field_value

FROM    physionet-data.mimiciv_note.radiology rad

        INNER JOIN physionet-data.mimiciv_note.radiology_detail det
        ON rad.note_id = det.note_id
"""
df_rad = client.query(radiology_query).to_dataframe()

print(df_rad.shape)
print(df_rad["field_name"].value_counts())   # audit all field types before filtering

df_rad["charttime"] = pd.to_datetime(df_rad["charttime"], utc=True)
df_rad["storetime"] = pd.to_datetime(df_rad["storetime"], utc=True)

(6021832, 6)
field_name
exam_code           2902561
exam_name           2902561
cpt_code             165504
parent_note_id        25603
addendum_note_id      25603
Name: count, dtype: int64


In [5]:
# Filter to exam_name rows and clean
df_rad = df_rad[df_rad["field_name"] == "exam_name"].copy()
df_rad["exam_name"] = df_rad["field_value"].str.upper().str.strip()

# Drop administrative and billing rows — these are not imaging exams
admin_patterns = [
    "BY SAME PHYSICIAN", "BY DIFFERENT PHYSICIAN",
    "DISTINCT PROCEDURAL SERVICE", "SEPARATE STRUCTURE",
    "MOD SEDATION", "FEE ADJUSTED", "___", "CAD "
]
mask_admin = df_rad["exam_name"].str.contains("|".join(admin_patterns), na=False)
df_rad = df_rad[~mask_admin].copy()

print(f"\nRows after filtering to exam_name and removing admin: {len(df_rad)}")


Rows after filtering to exam_name and removing admin: 2623806


In [6]:
# Resolve field_value format
# field_value can be one of three formats:
#   a) CPT code       — purely numeric, e.g. "71046"
#   b) Exam code      — short uppercase label, e.g. "CHEST (PORTABLE AP)"
#   c) Literal name   — verbose, e.g. "COMPUTED TOMOGRAPHY CHEST W/CONTRAST"
#
# CPT codes fall through string matching silently, landing in 'Other'.
# The lookup below maps common radiology CPT codes to (modality, body_region)
# before string matching runs, preventing that silent failure.

cpt_lookup = {
    # Chest X-ray
    "71045": ("X-ray",      "Chest"),
    "71046": ("X-ray",      "Chest"),
    "71047": ("X-ray",      "Chest"),
    "71048": ("X-ray",      "Chest"),
    # CT Chest
    "71250": ("CT",         "Chest"),
    "71260": ("CT",         "Chest"),
    "71270": ("CT",         "Chest"),
    # CTA Chest
    "71275": ("CTA",        "Chest"),
    # CT Abdomen/Pelvis
    "74150": ("CT",         "Abdomen"),
    "74160": ("CT",         "Abdomen"),
    "74170": ("CT",         "Abdomen"),
    "74177": ("CT",         "Abdomen"),
    "74178": ("CT",         "Abdomen"),
    "74179": ("CT",         "Abdomen"),
    "72192": ("CT",         "Pelvis/GYN"),
    "72193": ("CT",         "Pelvis/GYN"),
    "72194": ("CT",         "Pelvis/GYN"),
    # CT Head/Brain
    "70450": ("CT",         "Head/Neuro"),
    "70460": ("CT",         "Head/Neuro"),
    "70470": ("CT",         "Head/Neuro"),
    # CTA Head/Neck
    "70496": ("CTA",        "Head/Neuro"),
    "70498": ("CTA",        "Head/Neuro"),
    # MRI Brain
    "70551": ("MRI",        "Head/Neuro"),
    "70552": ("MRI",        "Head/Neuro"),
    "70553": ("MRI",        "Head/Neuro"),
    # MRI Spine
    "72141": ("MRI",        "Spine"),
    "72146": ("MRI",        "Spine"),
    "72148": ("MRI",        "Spine"),
    # Ultrasound Abdomen
    "76700": ("Ultrasound", "Abdomen"),
    "76705": ("Ultrasound", "Abdomen"),
    # Ultrasound Pelvis
    "76856": ("Ultrasound", "Pelvis/GYN"),
    "76857": ("Ultrasound", "Pelvis/GYN"),
    # Echocardiogram
    "93306": ("Ultrasound", "Cardiac"),
    "93307": ("Ultrasound", "Cardiac"),
    "93308": ("Ultrasound", "Cardiac"),
    # Fluoroscopy / line placement
    "77001": ("Fluoroscopy", "Vascular"),
    "77002": ("Fluoroscopy", "Vascular"),
}

def is_cpt_code(value):
    return bool(re.match(r"^\d{5}$", str(value).strip()))

In [7]:
# Assign modality and body region
# Resolution order: CPT lookup → string matching → fallback 'Other'

def assign_modality(name):
    if is_cpt_code(name):
        return cpt_lookup.get(name.strip(), ("Other", "Other"))[0]
    if "CTA " in name:
        return "CTA"
    if any(x in name for x in ["CT ", " CT", "CT ABD", "CT HEAD",
                                "CT CHEST", "CT C-SPINE", "CT PELVIS",
                                "COMPUTED TOMOGRAPHY"]):
        return "CT"
    if any(x in name for x in ["MR ", "MRI", " MR ", "MAGNETIC RESONANCE"]):
        return "MRI"
    if any(x in name for x in ["PORTABLE", "PORT.", "PORT ", "PA & LAT",
                                "AP & LAT", "AP,LAT", "PRE-OP", "SUPINE",
                                "CHEST (", "RADIOGRAPH", "X-RAY"]):
        return "X-ray"
    if any(x in name for x in [" US", "US.", "ULTRASOUND", "VEINS",
                                "RENAL U", "PELVIS U", "ECHO"]):
        return "Ultrasound"
    if any(x in name for x in ["FLUORO", "LINE PLACEMENT", "GUIDANCE", "PLCT"]):
        return "Fluoroscopy"
    if any(x in name for x in ["NUCLEAR", "SCAN", "PET"]):
        return "Nuclear"
    return "Other"

def assign_body_region(name):
    if is_cpt_code(name):
        return cpt_lookup.get(name.strip(), ("Other", "Other"))[1]
    if any(x in name for x in ["CHEST", "THORAC", "PULM"]):
        return "Chest"
    if any(x in name for x in ["HEAD", "BRAIN", "NEURO", "C-SPINE",
                                "NECK", "CRANIAL", "CEREBR"]):
        return "Head/Neuro"
    if any(x in name for x in ["ABD", "ABDOMEN", "LIVER", "GALLBLADDER",
                                "RENAL", "KIDNEY", "PANCREA", "BILIARY"]):
        return "Abdomen"
    if any(x in name for x in ["PELVIS", "PELVIC", "TRANSVAGINAL", "OB ",
                                "OBSTETRIC", "GYNECOL", "UTERUS", "OVARY"]):
        return "Pelvis/GYN"
    if any(x in name for x in ["SPINE", "L-SPINE", "T-SPINE",
                                "LUMBAR", "THORACIC SPINE"]):
        return "Spine"
    if any(x in name for x in ["CARDIAC", "CORONARY", "HEART", "ECHO"]):
        return "Cardiac"
    if any(x in name for x in ["VASCULAR", "AORTA", "VEINS",
                                "ARTERIO", "CTA"]):
        return "Vascular"
    if any(x in name for x in ["FOOT", "ANKLE", "KNEE", "HIP", "SHOULDER",
                                "ELBOW", "WRIST", "HAND", "FINGER",
                                "LOWER EXT", "UPPER EXT", "FEMUR", "TIBIA"]):
        return "Extremity"
    if any(x in name for x in ["BREAST", "MAMMOGRAM"]):
        return "Breast"
    return "Other"

df_rad["modality"]    = df_rad["exam_name"].apply(assign_modality)
df_rad["body_region"] = df_rad["exam_name"].apply(assign_body_region)

# Audit how many exam_name values still fall through to 'Other' after CPT lookup
other_modality = df_rad[df_rad["modality"] == "Other"]["exam_name"].value_counts()
print(f"\nUnresolved modality (Other): {len(other_modality)} unique values")
print(other_modality.head(20))   # review and extend cpt_lookup or string rules as needed


Unresolved modality (Other): 542 unique values
exam_name
PELVIS, NON-OBSTETRIC                      32581
DIG SCREENING BILAT                        13784
ANKLE (AP, MORTISE & LAT) RIGHT            13010
US GUID FOR VAS. ACCESS                    12376
ANKLE (AP, MORTISE & LAT) LEFT             12278
DIG DIAGNOSTIC MAMMO BILATERAL             11384
WRIST(3 + VIEWS) LEFT                      11370
PELVIS (AP ONLY)                           11160
KNEE (AP, LAT & OBLIQUE) RIGHT             10742
WRIST(3 + VIEWS) RIGHT                     10364
THYROID U.S.                               10249
KNEE (AP, LAT & OBLIQUE) LEFT              10100
CAROTID SERIES COMPLETE                     9612
MRA BRAIN W/O CONTRAST                      9601
ABDOMEN U.S. (COMPLETE STUDY)               9497
HIP UNILAT MIN 2 VIEWS RIGHT                9412
DUPLEX DOPP ABD/PEL                         9216
HIP UNILAT MIN 2 VIEWS LEFT                 9034
C-SPINE NON-TRAUMA 2-3 VIEWS                8880
PARACENTESI

In [8]:
# Flag high-acuity exams
# Covers all three formats: CPT codes, exam codes, and literal names
high_acuity_cpt = {
    "71250", "71260", "71270", "71275",   # CT/CTA chest
    "74177", "74178",                     # CT abdomen/pelvis
    "70450", "70460", "70470",            # CT head
    "70496", "70498",                     # CTA head/neck
    "70551", "70552", "70553",            # MRI brain
    "77001", "77002",                     # line placement
}

high_acuity_strings = [
    "CT HEAD", "CT CHEST", "CT ABD", "CT PELVIS",
    "CTA CHEST", "CTA HEAD", "CTA NECK",
    "MR HEAD", "MRI BRAIN",
    "CHEST (PORTABLE AP)", "CHEST PORT",
    "PORTABLE CHEST", "PORTACATH",
    "COMPUTED TOMOGRAPHY HEAD",
    "COMPUTED TOMOGRAPHY CHEST",
    "FLUORO GUID", "LINE PLACEMENT",
]

def is_high_acuity(name):
    if is_cpt_code(name):
        return int(name.strip() in high_acuity_cpt)
    return int(any(h in name for h in high_acuity_strings))

df_rad["is_high_acuity_exam"] = df_rad["exam_name"].apply(is_high_acuity)

In [9]:
# Aggregate to one row per patient
agg_rad = df_rad.groupby("subject_id").agg(

    # volume
    n_rad_notes          = ("note_id",            "nunique"),
    n_rad_exams          = ("exam_name",           "count"),
    n_high_acuity_exams  = ("is_high_acuity_exam", "sum"),
    has_high_acuity_exam = ("is_high_acuity_exam", "max"),

    # modality flags
    had_ct               = ("modality", lambda x: int("CT"          in x.values)),
    had_cta              = ("modality", lambda x: int("CTA"         in x.values)),
    had_mri              = ("modality", lambda x: int("MRI"         in x.values)),
    had_xray             = ("modality", lambda x: int("X-ray"       in x.values)),
    had_ultrasound       = ("modality", lambda x: int("Ultrasound"  in x.values)),
    had_fluoro           = ("modality", lambda x: int("Fluoroscopy" in x.values)),
    had_nuclear          = ("modality", lambda x: int("Nuclear"     in x.values)),

    # body region flags
    had_chest_imaging    = ("body_region", lambda x: int("Chest"      in x.values)),
    had_neuro_imaging    = ("body_region", lambda x: int("Head/Neuro" in x.values)),
    had_abdomen_imaging  = ("body_region", lambda x: int("Abdomen"    in x.values)),
    had_cardiac_imaging  = ("body_region", lambda x: int("Cardiac"    in x.values)),
    had_vascular_imaging = ("body_region", lambda x: int("Vascular"   in x.values)),
    had_spine_imaging    = ("body_region", lambda x: int("Spine"      in x.values)),
    had_extremity_imaging= ("body_region", lambda x: int("Extremity"  in x.values)),

    # timing — retained for potential interval extension
    first_exam_time      = ("charttime", "min"),
    last_exam_time       = ("charttime", "max"),

).reset_index()

# Exam span: hours between first and last radiology exam
# A wide span suggests prolonged monitoring; a narrow span suggests acute workup
agg_rad["exam_span_hrs"] = (
    (agg_rad["last_exam_time"] - agg_rad["first_exam_time"])
    .dt.total_seconds() / 3600
).round(2)

In [10]:
# Final table
# Drop timestamps from export — retained above only for span calculation
# and future interval work
rad_final = agg_rad.drop(columns=["first_exam_time", "last_exam_time"]).copy()

print(rad_final.shape)
print(rad_final.head())
print(rad_final.describe())

(236847, 20)
   subject_id  n_rad_notes  n_rad_exams  n_high_acuity_exams  \
0    10000032           24           25                    3   
1    10000084            4            4                    3   
2    10000102            1            1                    0   
3    10000108            1            1                    0   
4    10000117           18           20                    1   

   has_high_acuity_exam  had_ct  had_cta  had_mri  had_xray  had_ultrasound  \
0                     1       1        0        0         1               1   
1                     1       1        0        0         1               0   
2                     0       0        0        0         1               0   
3                     0       1        0        0         0               0   
4                     1       1        0        0         1               1   

   had_fluoro  had_nuclear  had_chest_imaging  had_neuro_imaging  \
0           0            0                  1              